# 3D Breast Surface Reconstruction — BreastNet3D (Fixed)
Self-supervised 3D reconstruction from 5-view DMR-IR thermal images.
Picks up from `breast_segmentation_unet_gpu.pth` — do NOT run the pip cell if packages are already installed.

**Fixes applied vs previous version:**
- `NameError`: `lr=optimizer.param_groups[0]['lr']` → `lr=config['lr']`
- `GradScaler` missing device string → `torch.amp.GradScaler('cuda', ...)`
- `DataLoader` missing `num_workers=0` → Windows multiprocessing deadlock fixed
- Grad checkpointing added to Decoder3D stages 4/5/6
- Direct execution cell at bottom — training starts immediately

In [1]:
# Run once only — skip if packages already installed
# !pip install torch torchvision tifffile opencv-python numpy scipy scikit-image matplotlib tqdm pandas

In [2]:
import os, sys, argparse, glob, json, math, time, random
import cv2
import tifffile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.ndimage
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict
from io import BytesIO

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory // 1024**3}GB")

PyTorch 2.13.0.dev20260420+cu126 | CUDA available: True
GPU: NVIDIA GeForce RTX 3060 | VRAM: 11GB


In [3]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu132

Looking in indexes: https://download.pytorch.org/whl/cu132
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement torchaudio (from versions: none)
ERROR: No matching distribution found for torchaudio


In [4]:
import torch
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")

12.6
True
1
NVIDIA GeForce RTX 3060


## Section 0: U-Net Architecture
Verbatim copy — must match `breast_segmentation_unet_gpu.pth` exactly.

In [5]:
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c, dropout=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Dropout2d(dropout) if dropout > 0 else nn.Identity(),
        )
    def forward(self, x): return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base_channels=64, dropout=0.2):
        super().__init__()
        b = base_channels
        self.pool = nn.MaxPool2d(2)
        self.enc1 = DoubleConv(in_channels, b,    dropout=0.0)
        self.enc2 = DoubleConv(b,    b*2,  dropout=0.0)
        self.enc3 = DoubleConv(b*2,  b*4,  dropout=0.1)
        self.enc4 = DoubleConv(b*4,  b*8,  dropout=0.1)
        self.bottleneck = DoubleConv(b*8, b*16, dropout=dropout)
        self.up4 = nn.ConvTranspose2d(b*16, b*8, 2, stride=2)
        self.dec4 = DoubleConv(b*16, b*8,  dropout=0.1)
        self.up3 = nn.ConvTranspose2d(b*8, b*4, 2, stride=2)
        self.dec3 = DoubleConv(b*8,  b*4,  dropout=0.1)
        self.up2 = nn.ConvTranspose2d(b*4, b*2, 2, stride=2)
        self.dec2 = DoubleConv(b*4,  b*2,  dropout=0.0)
        self.up1 = nn.ConvTranspose2d(b*2, b,   2, stride=2)
        self.dec1 = DoubleConv(b*2,  b,    dropout=0.0)
        self.out  = nn.Conv2d(b, out_channels, 1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b),  e4], 1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.out(d1)  # logits, no sigmoid

print("UNet defined.")

UNet defined.


## Section 1: Patient Grouper
Run the **DIAGNOSTIC cell first** if any patients show 4/5 views — it prints every unmatched filename so you can see exactly what pattern is failing.

In [6]:
# ── DIAGNOSTIC: run this once to see every filename in the dataset ────────────
# Shows exactly which filenames fail to match any view key.
# Read the output, then check / update VIEW_PATTERNS below if needed.

def diagnose_filenames(tiff_base):
    tiff_base = Path(tiff_base)
    matched   = {"RL": [], "RO": [], "F": [], "LO": [], "LL": []}
    unmatched = []

    # collect all image files (both .tiff and .tif)
    all_files = list(tiff_base.rglob("*.tiff")) + list(tiff_base.rglob("*.tif"))

    for p in sorted(all_files):
        name = p.name
        low  = name.lower()
        if   any(kw in low for kw in ["right lat", "r_lat", "rl"])   : matched["RL"].append(name)
        # FIXED — use same patterns as VIEW_PATTERNS
        elif any(kw in low for kw in ["right obl", "right_obl", "r_obl", "_ro.", "_ro_"]): matched["RO"].append(name)
        elif any(kw in low for kw in ["frontal", "front", "anterior", "_f.", "_f_", "(0"]): matched["F"].append(name)
        elif any(kw in low for kw in ["left obl",  "l_obl",  "lo"])  : matched["LO"].append(name)
        elif any(kw in low for kw in ["left lat",  "l_lat",  "ll"])  : matched["LL"].append(name)
        else:
            unmatched.append(str(p.relative_to(tiff_base)))

    print("=" * 60)
    print("MATCHED EXAMPLES (first 3 per view):")
    for k, v in matched.items():
        examples = list(dict.fromkeys(v))[:3]  # unique, first 3
        print(f"  {k}: {examples}")
    print()
    if unmatched:
        print(f"UNMATCHED FILES ({len(unmatched)} total — these cause the 4/5 warning):")
        for f in sorted(set(unmatched))[:30]:
            print(f"  {f}")
    else:
        print("No unmatched files. All filenames recognised.")
    print("=" * 60)

# ── run diagnostic ────────────────────────────────────────────────────────────
TIFF_BASE = r"C:\Users\User\Documents\Skripsi Faiz_DL\DNP-3DDMR-IR\data\organized_by_patient"
diagnose_filenames(TIFF_BASE)

MATCHED EXAMPLES (first 3 per view):
  RL: ['Right Lateral (90°).tiff']
  RO: ['Right Oblique (45°).tiff']
  F: ['Anterior (Front).tiff']
  LO: ['Left Oblique (45°).tiff']
  LL: ['Left Lateral (90°).tiff']

No unmatched files. All filenames recognised.


In [7]:
@dataclass
class PatientGroup:
    patient_id: str
    label: str
    views: Dict[str, Path]
    masks: Dict[str, Path]

# ── VIEW_PATTERNS: edit these if diagnose_filenames() reveals non-standard names
# Each entry is a list of lowercase substrings — ANY match → that view key.
VIEW_PATTERNS = {
    "RL": ["right lat", "right_lat", "r_lat", "_rl.", "_rl_"],
    "RO": ["right obl", "right_obl", "r_obl", "_ro.", "_ro_"],
    "F" : ["frontal",   "front",     "_f.",   "_f_",  "(0"],
    "LO": ["left obl",  "left_obl",  "l_obl", "_lo.", "_lo_"],
    "LL": ["left lat",  "left_lat",  "l_lat", "_ll.", "_ll_"],
}

def get_view_key(filename: str) -> str | None:
    """Robust view classifier — matches substrings anywhere in lowercase filename."""
    low = filename.lower()
    for key, patterns in VIEW_PATTERNS.items():
        if any(pat in low for pat in patterns):
            return key
    return None

def build_patient_groups(tiff_base, mask_base) -> List[PatientGroup]:
    tiff_base = Path(tiff_base)
    mask_base = Path(mask_base)
    patient_dict = {}

    # scan both .tiff and .tif
    image_files = list(tiff_base.rglob("*.tiff")) + list(tiff_base.rglob("*.tif"))

    for tiff_path in image_files:
        rel   = tiff_path.relative_to(tiff_base)
        parts = rel.parts
        if len(parts) < 3:
            continue
        patient_id = parts[0]
        label      = parts[1]
        view_key   = get_view_key(parts[-1])
        if not view_key:
            continue
        key = (patient_id, label)
        if key not in patient_dict:
            patient_dict[key] = {"views": {}, "masks": {}}
        # keep first match per view per patient (no duplicate overwrite)
        if view_key not in patient_dict[key]["views"]:
            patient_dict[key]["views"][view_key] = tiff_path

        mask_dir = mask_base / patient_id / label
        if mask_dir.exists() and view_key not in patient_dict[key]["masks"]:
            for mf in mask_dir.iterdir():
                if get_view_key(mf.name) == view_key:
                    patient_dict[key]["masks"][view_key] = mf
                    break

    complete, n_incomplete, n_benign, n_mal = [], 0, 0, 0
    for (pid, lbl), data in sorted(patient_dict.items()):
        found = set(data["views"].keys())
        if len(found) == 5:
            complete.append(PatientGroup(pid, lbl, data["views"], data["masks"]))
            if lbl.lower() == "benign": n_benign += 1
            else:                        n_mal    += 1
        else:
            missing = set(["RL","RO","F","LO","LL"]) - found
            print(f"  WARNING: {pid} ({lbl}) — {len(found)}/5 views  missing: {missing}")
            n_incomplete += 1

    complete.sort(key=lambda x: x.patient_id)
    print(f"Total patients scanned  : {len(patient_dict)}")
    print(f"Complete (5-view) groups: {len(complete)}")
    print(f"Incomplete (skipped)    : {n_incomplete}")
    print(f"Class split             : Benign={n_benign}  Malignant={n_mal}")
    return complete

## Section 2: Patient Dataset

In [8]:
class PatientDataset(Dataset):
    def __init__(self, groups: List[PatientGroup], unet: UNet, device, img_size=256):
        self.groups     = groups
        self.unet       = unet
        self.device     = device
        self.img_size   = img_size
        self.view_order = ["RL", "RO", "F", "LO", "LL"]

    def __len__(self): return len(self.groups)

    def __getitem__(self, idx):
        group    = self.groups[idx]
        thermals = []
        masks    = []

        for view in self.view_order:
            # --- thermal ---
            raw  = tifffile.imread(str(group.views[view])).astype(np.float32)
            raw  = cv2.resize(raw, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
            norm = (raw - raw.min()) / (raw.max() - raw.min() + 1e-8)
            thermals.append(norm)

            # --- mask ---
            mp = group.masks.get(view)
            if mp and Path(mp).exists():
                arr  = np.fromfile(str(mp), dtype=np.uint8)          # Windows degree-symbol safe
                img  = cv2.imdecode(arr, cv2.IMREAD_GRAYSCALE)
                img  = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
                mask = (img / 255.0 > 0.5).astype(np.float32)
            else:
                with torch.no_grad():
                    inp  = torch.tensor(norm).unsqueeze(0).unsqueeze(0).to(self.device)
                    mask = (torch.sigmoid(self.unet(inp)).squeeze().cpu().numpy() > 0.5).astype(np.float32)

            # resize mask to 128x128 for the 3D network
            mask128 = cv2.resize(mask, (128, 128), interpolation=cv2.INTER_NEAREST)
            masks.append(mask128)

        return {
            "masks_5ch"   : torch.tensor(np.stack(masks,    axis=0), dtype=torch.float32),
            "thermals_5ch": torch.tensor(np.stack(thermals, axis=0), dtype=torch.float32),
            "patient_id"  : group.patient_id,
            "label"       : group.label,
            "view_order"  : self.view_order,
        }

## Section 3: Encoder2D

In [9]:
def _init_he(m):
    if isinstance(m, (nn.Conv2d, nn.Conv3d, nn.ConvTranspose2d, nn.ConvTranspose3d)):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None: nn.init.constant_(m.bias, 0)
    elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm3d)):
        nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None: nn.init.constant_(m.bias, 0)

class DoubleConv2D(nn.Module):
    def __init__(self, in_c, out_c, dropout=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Dropout2d(dropout) if dropout > 0 else nn.Identity(),
        )
    def forward(self, x): return self.block(x)

class Encoder2D(nn.Module):
    """5-channel 128x128 masks → 1000-dim latent code."""
    def __init__(self, dropout=0.25):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        # 128 → 64 → 32 → 16 → 8 → 4 → 2
        self.enc1 = DoubleConv2D(5,   32,  dropout=0.0)
        self.enc2 = DoubleConv2D(32,  64,  dropout=0.0)
        self.enc3 = DoubleConv2D(64,  128, dropout=dropout)
        self.enc4 = DoubleConv2D(128, 256, dropout=dropout)
        self.enc5 = DoubleConv2D(256, 512, dropout=dropout)
        self.enc6 = DoubleConv2D(512, 512, dropout=dropout)  # extra stage for 128px input
        self.fc   = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(512 * 2 * 2, 1000)
        )
        self.apply(_init_he)

    def forward(self, x):
        # x: [B, 5, 128, 128]
        x = self.enc1(x);           # [B,  32, 128, 128]
        x = self.enc2(self.pool(x)) # [B,  64,  64,  64]
        x = self.enc3(self.pool(x)) # [B, 128,  32,  32]
        x = self.enc4(self.pool(x)) # [B, 256,  16,  16]
        x = self.enc5(self.pool(x)) # [B, 512,   8,   8]
        x = self.enc6(self.pool(x)) # [B, 512,   4,   4]
        x = self.pool(x)            # [B, 512,   2,   2]
        x = x.view(x.size(0), -1)   # [B, 2048]
        return self.fc(x)           # [B, 1000]

print(f"Encoder2D params: {sum(p.numel() for p in Encoder2D().parameters()):,}")

Encoder2D params: 11,484,424


## Section 4: Decoder3D (with gradient checkpointing)

In [10]:
class DoubleConv3D(nn.Module):
    def __init__(self, in_c, out_c, dropout=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm3d(out_c), nn.ReLU(inplace=True),
            nn.Conv3d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm3d(out_c), nn.ReLU(inplace=True),
            nn.Dropout3d(dropout) if dropout > 0 else nn.Identity(),
        )
    def forward(self, x): return self.block(x)

class Decoder3D(nn.Module):
    """1000-dim latent → [B, 1, 128, 128, 128] voxel volume.
    Stages 4/5/6 use gradient checkpointing to save VRAM on RTX 3060.
    """
    def __init__(self, dropout=0.25, use_checkpoint=True):
        super().__init__()
        self.use_checkpoint = use_checkpoint

        self.fc = nn.Linear(1000, 512 * 2 * 2 * 2)   # → reshape [B,512,2,2,2]

        # 2³ → 4³ → 8³ → 16³ → 32³ → 64³ → 128³
        self.up1  = nn.ConvTranspose3d(512, 256, 2, stride=2)
        self.dec1 = DoubleConv3D(256, 256, dropout=dropout)

        self.up2  = nn.ConvTranspose3d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv3D(128, 128, dropout=dropout)

        self.up3  = nn.ConvTranspose3d(128, 64, 2, stride=2)
        self.dec3 = DoubleConv3D(64,  64,  dropout=dropout)

        # stages 4-6: expensive feature maps — use grad checkpoint
        self.up4  = nn.ConvTranspose3d(64, 32, 2, stride=2)
        self.dec4 = DoubleConv3D(32, 32, dropout=0.0)

        self.up5  = nn.ConvTranspose3d(32, 16, 2, stride=2)
        self.dec5 = DoubleConv3D(16, 16, dropout=0.0)

        self.up6  = nn.ConvTranspose3d(16, 8, 2, stride=2)
        self.dec6 = DoubleConv3D(8,  8,  dropout=0.0)

        self.out  = nn.Sequential(nn.Conv3d(8, 1, 1), nn.Sigmoid())
        self.apply(_init_he)

    # --- helper methods for checkpointing ---
    def _stage4(self, x): return self.dec4(self.up4(x))
    def _stage5(self, x): return self.dec5(self.up5(x))
    def _stage6(self, x): return self.dec6(self.up6(x))

    def forward(self, x):
        x = self.fc(x)
        x = x.view(x.size(0), 512, 2, 2, 2)   # 2³
        x = self.dec1(self.up1(x))             # 4³
        x = self.dec2(self.up2(x))             # 8³
        x = self.dec3(self.up3(x))             # 16³

        if self.use_checkpoint and self.training:
            x = checkpoint(self._stage4, x, use_reentrant=False)  # 32³
            x = checkpoint(self._stage5, x, use_reentrant=False)  # 64³
            x = checkpoint(self._stage6, x, use_reentrant=False)  # 128³
        else:
            x = self._stage4(x)  # 32³
            x = self._stage5(x)  # 64³
            x = self._stage6(x)  # 128³

        return self.out(x)  # [B, 1, 128, 128, 128]

print(f"Decoder3D params: {sum(p.numel() for p in Decoder3D().parameters()):,}")

Decoder3D params: 10,217,825


## Section 5: Differentiable Visual Hull Renderer

In [18]:
def render_projection(volume, theta_deg):
    """Differentiable 3D→2D projection via Y-axis rotation + visual hull.

    Args:
        volume    : [B, 1, D, H, W]  float tensor in [0,1]
        theta_deg : scalar or [B] tensor, degrees
    Returns:
        projection: [B, 1, H, W]  float tensor in [0,1]
    """
    B, C, D, H, W = volume.shape
    device, dtype = volume.device, volume.dtype

    if not isinstance(theta_deg, torch.Tensor):
        theta_deg = torch.full((B,), float(theta_deg), device=device, dtype=dtype)
    elif theta_deg.dim() == 0:
        theta_deg = theta_deg.unsqueeze(0).expand(B)

    theta_rad = theta_deg * (math.pi / 180.0)
    cos_t = torch.cos(theta_rad)          # [B]
    sin_t = torch.sin(theta_rad)          # [B]
    zero  = torch.zeros_like(theta_rad)
    one   = torch.ones_like(theta_rad)

    # Ry(theta) rotation matrix  [B, 3, 4]
    theta_matrix = torch.stack([
        torch.stack([ cos_t, zero,  sin_t, zero], dim=-1),
        torch.stack([ zero,  one,   zero,  zero], dim=-1),
        torch.stack([-sin_t, zero,  cos_t, zero], dim=-1),
    ], dim=-2)  # [B, 3, 4]

    grid  = F.affine_grid(theta_matrix, volume.shape, align_corners=False)
    grid  = grid.to(dtype=volume.dtype, device=volume.device)  # Ensure grid matches volume dtype/device
    V_rot = F.grid_sample(volume, grid, mode='bilinear', padding_mode='zeros', align_corners=False)

    # Visual hull: collapse depth axis  [B, D, H, W] → [B, 1, H, W]
    v = V_rot.squeeze(1)                                  # [B, D, H, W]
    projection = 1.0 - torch.exp(-v.sum(dim=1, keepdim=True))  # [B, 1, H, W]

    return projection

print("render_projection defined.")

render_projection defined.


## Section 6: Self-Supervised Training

In [19]:
def dice_loss(pred, target, eps=1e-6):
    """Soft Dice loss — fully differentiable, no thresholding."""
    num = 2.0 * (pred * target).sum()
    den = pred.pow(2).sum() + target.pow(2).sum() + eps
    return 1.0 - num / den

def hd95(p, t):
    """95th-percentile Hausdorff distance between two binary 2D arrays."""
    if p.sum() == 0 or t.sum() == 0:
        return 128.0
    p_edges = p ^ scipy.ndimage.binary_erosion(p)
    t_edges = t ^ scipy.ndimage.binary_erosion(t)
    dt_p = scipy.ndimage.distance_transform_edt(~p_edges)
    dt_t = scipy.ndimage.distance_transform_edt(~t_edges)
    d1 = np.percentile(dt_t[p_edges], 95) if p_edges.sum() > 0 else 128.0
    d2 = np.percentile(dt_p[t_edges], 95) if t_edges.sum() > 0 else 128.0
    return float(max(d1, d2))

VIEW_WINDOWS = [
    (-90.0, -67.5),   # RL
    (-67.5, -22.5),   # RO
    (-22.5,  22.5),   # F
    ( 22.5,  67.5),   # LO
    ( 67.5,  90.0),   # LL
]
VAL_ANGLES = [-90.0, -45.0, 0.0, 45.0, 90.0]


def train_3dbreastnet(config: dict):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    set_seed(config['seed'])

    # --- load frozen U-Net ---
    unet = UNet().to(device)
    unet.load_state_dict(torch.load(config['unet_ckpt'], map_location=device))
    unet.eval()
    for p in unet.parameters(): p.requires_grad = False
    print(f"U-Net loaded from {config['unet_ckpt']}")

    # --- build groups and stratified split ---
    groups   = build_patient_groups(config['tiff_base'], config['mask_base'])
    rng      = random.Random(config['seed'])
    benign   = [g for g in groups if g.label.lower() == 'benign']
    malignt  = [g for g in groups if g.label.lower() != 'benign']
    rng.shuffle(benign); rng.shuffle(malignt)
    split_b  = int(len(benign)  * 0.78)
    split_m  = int(len(malignt) * 0.78)
    train_groups = benign[:split_b]  + malignt[:split_m]
    val_groups   = benign[split_b:]  + malignt[split_m:]
    print(f"Train: {len(train_groups)} patients | Val: {len(val_groups)} patients")

    if len(train_groups) == 0:
        raise RuntimeError("No complete 5-view patients found. Check TIFF_BASE path.")

    # --- datasets and loaders ---
    train_ds = PatientDataset(train_groups, unet, device)
    val_ds   = PatientDataset(val_groups,   unet, device)
    # num_workers=0 is REQUIRED on Windows in a Jupyter notebook
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'],
                              shuffle=True,  drop_last=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=config['batch_size'],
                              shuffle=False, drop_last=False, num_workers=0)

    # --- models ---
    encoder = Encoder2D().to(device)
    decoder = Decoder3D(use_checkpoint=config.get('use_grad_checkpoint', True)).to(device)
    params  = list(encoder.parameters()) + list(decoder.parameters())

    # FIX: was `lr=optimizer.param_groups[0]['lr']` (NameError — optimizer didn't exist yet)
    optimizer = torch.optim.Adam(params, lr=config['lr'], betas=config['betas'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=20, min_lr=1e-6
    )
    # FIX: was torch.amp.GradScaler(enabled=...) — missing 'cuda' device string
    scaler = torch.amp.GradScaler("cuda", enabled=config['use_amp'])

    Path(config['ckpt_dir']).mkdir(parents=True, exist_ok=True)

    best_val_dice    = 0.0
    epochs_no_improv = 0
    patience         = config.get('patience', 40)
    history = {'epoch': [], 'train_loss': [], 'val_loss': [], 'val_dice': [], 'val_hd': []}

    print(f"Starting training for {config['epochs']} epochs...")
    for epoch in range(1, config['epochs'] + 1):
        t0 = time.time()
        encoder.train(); decoder.train()
        train_loss_acc = 0.0

        for batch in train_loader:
            masks = batch['masks_5ch'].to(device)   # [B, 5, 128, 128]
            B     = masks.size(0)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", enabled=config['use_amp']):
                volume = decoder(encoder(masks))    # [B, 1, 128, 128, 128]
                loss   = 0.0
                for i, (lo, hi) in enumerate(VIEW_WINDOWS):
                    for _ in range(config['n_per_view']):
                        theta = torch.rand(B, device=device) * (hi - lo) + lo
                        proj  = render_projection(volume, theta)
                        loss += dice_loss(proj, masks[:, i:i+1])
                loss = loss / (5 * config['n_per_view'])

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            train_loss_acc += loss.item()

        train_loss = train_loss_acc / max(1, len(train_loader))

        # --- validation ---
        encoder.eval(); decoder.eval()
        vl_acc, vd_acc, vh_acc, count = 0.0, 0.0, 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                masks = batch['masks_5ch'].to(device)
                B     = masks.size(0)
                with torch.amp.autocast("cuda", enabled=config['use_amp']):
                    volume = decoder(encoder(masks))
                for i, ang in enumerate(VAL_ANGLES):
                    theta = torch.full((B,), ang, device=device)
                    proj  = render_projection(volume, theta)
                    vl    = dice_loss(proj, masks[:, i:i+1]).item()
                    vl_acc += vl
                    vd_acc += (1.0 - vl)
                    pb = (proj > 0.5).cpu().numpy()
                    mb = (masks[:, i:i+1] > 0.5).cpu().numpy()
                    for b in range(B):
                        vh_acc += hd95(pb[b, 0], mb[b, 0])
                        count  += 1

        n_val_terms = max(1, len(val_loader) * 5)
        val_loss = vl_acc / n_val_terms
        val_dice = vd_acc / n_val_terms
        val_hd   = vh_acc / max(1, count)
        elapsed  = time.time() - t0

        history['epoch'].append(epoch)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_dice'].append(val_dice)
        history['val_hd'].append(val_hd)

        scheduler.step(val_dice)
        lr_now = optimizer.param_groups[0]['lr']

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:03d} | train={train_loss:.4f} | val_dice={val_dice:.4f} "
                  f"| val_hd={val_hd:.2f} | lr={lr_now:.2e} | {elapsed:.1f}s")

        ckpt = {
            'epoch': epoch,
            'encoder_state': encoder.state_dict(),
            'decoder_state': decoder.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'best_val_dice': max(best_val_dice, val_dice),
            'config': config,
            'history': history,
        }
        torch.save(ckpt, Path(config['ckpt_dir']) / "3dbreastnet_last.pth")

        if val_dice > best_val_dice:
            best_val_dice    = val_dice
            epochs_no_improv = 0
            torch.save(ckpt, Path(config['ckpt_dir']) / "3dbreastnet_best.pth")
            print(f"  => New best val_dice: {best_val_dice:.4f}")
        else:
            epochs_no_improv += 1
            if epochs_no_improv >= patience:
                print(f"Early stopping at epoch {epoch} (no improvement for {patience} epochs).")
                break

    # --- plot ---
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(history['epoch'], history['train_loss'], label='train')
    axes[0].plot(history['epoch'], history['val_loss'],   label='val')
    axes[0].set_title('Dice loss'); axes[0].legend()
    axes[1].plot(history['epoch'], history['val_dice'], color='green')
    axes[1].set_title('Val Dice')
    axes[2].plot(history['epoch'], history['val_hd'], color='red')
    axes[2].set_title('Val HD95')
    plt.tight_layout()
    plt.savefig(Path(config['ckpt_dir']) / 'training_history.png', dpi=120)
    plt.show()
    print("Training complete.")

print("train_3dbreastnet defined.")

train_3dbreastnet defined.


## Section 7: Inference & Temperature Overlay

In [20]:
def run_inference(patient_group, encoder, decoder, unet, device):
    ds   = PatientDataset([patient_group], unet, device)
    item = ds[0]
    masks_5ch    = item['masks_5ch'].unsqueeze(0).to(device)  # [1,5,128,128]
    thermals_5ch = item['thermals_5ch'].numpy()               # [5,256,256]

    with torch.no_grad():
        volume = decoder(encoder(masks_5ch))   # [1,1,128,128,128]

        # --- find best angle per view (full sweep -90..+90 at 1° step) ---
        angles      = torch.arange(-90, 91, 1.0, dtype=torch.float32, device=device)
        best_angles = []
        for i in range(5):
            best_loss, best_a = float('inf'), angles[0].item()
            for a in angles:
                loss = dice_loss(
                    render_projection(volume, a),
                    masks_5ch[:, i:i+1]
                ).item()
                if loss < best_loss:
                    best_loss, best_a = loss, a.item()
            best_angles.append(best_a)

    vol_soft = volume[0, 0].cpu().numpy()          # float32 [128,128,128]
    vol_bin  = (vol_soft > 0.5).astype(np.uint8)

    # --- vectorised temperature overlay ---
    vol_thermal = torch.zeros((1, 1, 128, 128, 128), device=device)
    vol_counts  = torch.zeros((1, 1, 128, 128, 128), device=device)

    with torch.no_grad():
        for i, a in enumerate(best_angles):
            theta_rad = a * math.pi / 180.0
            cos_t, sin_t = math.cos(theta_rad), math.sin(theta_rad)

            fwd_mat = torch.tensor([[
                [ cos_t, 0.0,  sin_t, 0.0],
                [ 0.0,   1.0,  0.0,   0.0],
                [-sin_t, 0.0,  cos_t, 0.0],
            ]], device=device, dtype=torch.float32)
            inv_mat = torch.tensor([[
                [cos_t, 0.0, -sin_t, 0.0],
                [0.0,   1.0,  0.0,   0.0],
                [sin_t, 0.0,  cos_t, 0.0],
            ]], device=device, dtype=torch.float32)

            fwd_grid = F.affine_grid(fwd_mat, volume.shape, align_corners=False)
            V_rot    = F.grid_sample(volume, fwd_grid, mode='bilinear',
                                     padding_mode='zeros', align_corners=False)
            V_rot_bin = V_rot > 0.5   # [1,1,128,128,128]

            # thermal for this view resized to 128x128
            th_np  = cv2.resize(thermals_5ch[i], (128, 128), interpolation=cv2.INTER_LINEAR)
            th_t   = torch.tensor(th_np, device=device)  # [128,128]

            # vectorised scatter: find front-most occupied voxel per (h,w) column
            depth_indices = V_rot_bin[0, 0].float().argmax(dim=0)  # [128,128]
            has_val       = V_rot_bin[0, 0].any(dim=0)             # [128,128]

            T_rot = torch.zeros((1, 1, 128, 128, 128), device=device)
            C_rot = torch.zeros((1, 1, 128, 128, 128), device=device)

            h_idx, w_idx = has_val.nonzero(as_tuple=True)
            if h_idx.numel() > 0:
                d = depth_indices[h_idx, w_idx].long()
                T_rot[0, 0, d, h_idx, w_idx] = th_t[h_idx, w_idx]
                C_rot[0, 0, d, h_idx, w_idx] = 1.0

            # inverse-rotate back to canonical space
            inv_grid = F.affine_grid(inv_mat, volume.shape, align_corners=False)
            T_orig   = F.grid_sample(T_rot, inv_grid, mode='nearest',
                                     padding_mode='zeros', align_corners=False)
            C_orig   = F.grid_sample(C_rot, inv_grid, mode='nearest',
                                     padding_mode='zeros', align_corners=False)
            vol_thermal += T_orig
            vol_counts  += C_orig

        valid = vol_counts > 0
        vol_thermal[valid] /= vol_counts[valid]

        proj_5ch = [render_projection(volume, a)[0, 0].cpu().numpy() for a in best_angles]

    return {
        "volume_binary" : vol_bin,
        "volume_soft"   : vol_soft,
        "volume_thermal": vol_thermal[0, 0].cpu().numpy(),
        "estimated_angles": {k: v for k, v in zip(["RL","RO","F","LO","LL"], best_angles)},
        "patient_id"    : patient_group.patient_id,
        "masks_5ch"     : masks_5ch[0].cpu().numpy(),
        "proj_5ch"      : proj_5ch,
    }


def save_projection_check(out_dict, out_path):
    masks = out_dict["masks_5ch"]
    projs = out_dict["proj_5ch"]
    views = ["RL", "RO", "F", "LO", "LL"]
    fig, axes = plt.subplots(5, 3, figsize=(9, 15))
    for i in range(5):
        mask = masks[i]
        proj = (projs[i] > 0.5).astype(np.float32)
        diff = np.abs(mask - proj)
        axes[i, 0].imshow(mask, cmap='gray'); axes[i, 0].set_title(f"{views[i]} mask")
        axes[i, 1].imshow(proj, cmap='gray'); axes[i, 1].set_title(f"{views[i]} proj")
        axes[i, 2].imshow(diff, cmap='hot');  axes[i, 2].set_title(f"{views[i]} diff")
        for ax in axes[i]: ax.axis('off')
    plt.tight_layout()
    buf = BytesIO(); plt.savefig(buf, format='png'); plt.close(fig); buf.seek(0)
    arr = np.frombuffer(buf.getvalue(), dtype=np.uint8)
    img = cv2.imdecode(arr, 1)
    ok, buf2 = cv2.imencode('.png', img)
    if ok: buf2.tofile(str(out_path))

print("run_inference / save_projection_check defined.")

run_inference / save_projection_check defined.


## Section 8: Asymmetry Features

In [21]:
def extract_asymmetry_features(volume_binary, volume_thermal) -> dict:
    vol_voxels = int(np.sum(volume_binary))
    z, y, x = np.nonzero(volume_binary)
    if len(x) > 0:
        cx, cy, cz = float(np.mean(x)), float(np.mean(y)), float(np.mean(z))
        bbox_w = int(x.max() - x.min() + 1)
        bbox_h = int(y.max() - y.min() + 1)
        bbox_d = int(z.max() - z.min() + 1)
    else:
        cx = cy = cz = 0.0
        bbox_w = bbox_h = bbox_d = 0

    eroded          = scipy.ndimage.binary_erosion(volume_binary)
    surface         = (volume_binary > 0) & (~eroded)
    surface_voxels  = int(np.sum(surface))

    left_half_vol  = int(np.sum(volume_binary[:, :, :64]))
    right_half_vol = int(np.sum(volume_binary[:, :, 64:]))
    lr_asym        = abs(left_half_vol - right_half_vol) / (left_half_vol + right_half_vol + 1e-6)

    left_surf  = surface.copy(); left_surf[:, :, 64:]  = False
    right_surf = surface.copy(); right_surf[:, :, :64] = False
    tl = volume_thermal[left_surf]
    tr = volume_thermal[right_surf]
    ts = volume_thermal[surface]

    mean_tl = float(np.mean(tl)) if tl.size > 0 else 0.0
    mean_tr = float(np.mean(tr)) if tr.size > 0 else 0.0

    return {
        "volume_voxels"    : vol_voxels,
        "centroid_x"       : cx, "centroid_y": cy, "centroid_z": cz,
        "bbox_w"           : bbox_w, "bbox_h": bbox_h, "bbox_d": bbox_d,
        "surface_voxels"   : surface_voxels,
        "left_half_vol"    : left_half_vol,
        "right_half_vol"   : right_half_vol,
        "lr_asymmetry"     : float(lr_asym),
        "mean_temp_left"   : mean_tl,
        "mean_temp_right"  : mean_tr,
        "thermal_asymmetry": abs(mean_tl - mean_tr),
        "std_temp"         : float(np.std(ts))  if ts.size > 0 else 0.0,
        "max_temp"         : float(np.max(ts))  if ts.size > 0 else 0.0,
    }

print("extract_asymmetry_features defined.")

extract_asymmetry_features defined.


## Section 9: Run Training
Edit the paths below then run this cell. Training starts immediately.

In [ ]:
# ── EDIT THESE PATHS ──────────────────────────────────────────────────────────
TIFF_BASE  = r"C:\Users\User\Documents\Skripsi Faiz_DL\DNP-3DDMR-IR\data\organized_by_patient"
MASK_BASE  = r"C:\Users\User\Documents\Skripsi Faiz_DL\DNP-3DDMR-IR\data\GroundTruth_Masks"
UNET_CKPT  = r"C:\Users\User\Documents\Skripsi Faiz_DL\DNP-3DDMR-IR\UNET_Segmentation\breast_segmentation_unet_gpu.pth"
CKPT_DIR   = "checkpoints_3d"
# ──────────────────────────────────────────────────────────────────────────────

CONFIG = {
    'tiff_base'           : TIFF_BASE,
    'mask_base'           : MASK_BASE,
    'unet_ckpt'           : UNET_CKPT,
    'ckpt_dir'            : CKPT_DIR,
    'epochs'              : 400,
    'batch_size'          : 2,       # safe for RTX 3060 at 128³
    'lr'                  : 0.025,
    'betas'               : (0.5, 0.9),
    'n_per_view'          : 2,
    'seed'                : 42,
    'use_amp'             : torch.cuda.is_available(),
    'use_grad_checkpoint' : True,
    'patience'            : 40,      # early stopping patience
}

train_3dbreastnet(CONFIG)

Device: cuda
U-Net loaded from C:\Users\User\Documents\Skripsi Faiz_DL\DNP-3DDMR-IR\UNET_Segmentation\breast_segmentation_unet_gpu.pth
Total patients scanned  : 137
Complete (5-view) groups: 122
Incomplete (skipped)    : 15
Class split             : Benign=96  Malignant=26
Train: 94 patients | Val: 28 patients
Starting training for 400 epochs...
Epoch 001 | train=nan | val_dice=nan | val_hd=128.00 | lr=2.50e-02 | 19.5s


## Section 10: Run Inference (after training)
Loads best checkpoint, runs inference on all complete patients, exports volumes and features.

In [ ]:
OUT_3D_DIR = r"C:\Users\User\Documents\Skripsi Faiz_DL\DNP-3DDMR-IR\data\Volumes_3D"
BEST_CKPT  = f"{CKPT_DIR}/3dbreastnet_best.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

unet = UNet().to(device)
unet.load_state_dict(torch.load(UNET_CKPT, map_location=device))
unet.eval()

encoder = Encoder2D().to(device)
decoder = Decoder3D(use_checkpoint=False).to(device)
ckpt = torch.load(BEST_CKPT, map_location=device)
encoder.load_state_dict(ckpt['encoder_state'])
decoder.load_state_dict(ckpt['decoder_state'])
encoder.eval(); decoder.eval()
print(f"Loaded checkpoint from epoch {ckpt['epoch']} (best val_dice={ckpt['best_val_dice']:.4f})")

groups  = build_patient_groups(TIFF_BASE, MASK_BASE)
out_dir = Path(OUT_3D_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

features_list = []
for g in groups:
    print(f"  Inferring {g.patient_id}...")
    out = run_inference(g, encoder, decoder, unet, device)

    p_dir = out_dir / g.patient_id
    p_dir.mkdir(parents=True, exist_ok=True)
    np.save(str(p_dir / "volume_binary.npy"),  out["volume_binary"])
    np.save(str(p_dir / "volume_thermal.npy"), out["volume_thermal"])
    with open(p_dir / "estimated_angles.json", "w") as f:
        json.dump(out["estimated_angles"], f, indent=2)
    save_projection_check(out, p_dir / "projection_check.png")

    feats = extract_asymmetry_features(out["volume_binary"], out["volume_thermal"])
    feats["patient_id"] = g.patient_id
    feats["label"]      = g.label
    features_list.append(feats)

if features_list:
    df   = pd.DataFrame(features_list)
    cols = ["patient_id", "label"] + [c for c in df.columns if c not in ["patient_id", "label"]]
    df[cols].to_csv(out_dir / "asymmetry_features.csv", index=False)
    print(f"\nFeatures saved to {out_dir / 'asymmetry_features.csv'}")
    display(df[cols].head())